In [ ]:
# pip install -U google-generativeai gradio langchain langchain-community faiss-cpu sentence-transformers huggingface-hub
# If you're in a notebook environment, you may prefer:
# %pip install -U google-generativeai gradio langchain langchain-community faiss-cpu sentence-transformers huggingface-hub

%matplotlib inline


# Task 4 — RAG Chatbot (LangChain + FAISS)

## Objective
Build a small **Retrieval-Augmented Generation (RAG)** chatbot using LangChain, backed by a custom mini-corpus and a FAISS vector store.

## Knowledge Base
A tiny custom corpus (3–4 short Wikipedia-style paragraphs) about AI topics.

## LLM
- Primary: **Gemini 2.5 Flash** via `google-generativeai` (free tier)
- Fallback: Hugging Face free inference via `HuggingFaceEndpoint` (e.g., `mistralai/Mistral-7B-Instruct-v0.3`) with `HF_TOKEN`

## Approach
- Embed with `HuggingFaceEmbeddings` (`sentence-transformers/all-MiniLM-L6-v2`)
- Store vectors in FAISS
- Retrieve top-k passages for each user query
- Maintain conversation state with `ConversationBufferMemory` (trimmed to last 3 exchanges)
- Provide a Gradio chat UI

## Final Summary / Insights
RAG improves factuality by grounding responses on retrieved context. This notebook shows a minimal, free-to-run setup that works with either Gemini or a free HF endpoint.


In [ ]:
import os
import textwrap
from typing import List, Tuple

try:
    import google.generativeai as genai
except Exception:
    genai = None

try:
    import gradio as gr
except Exception as e:
    raise ImportError("gradio is required. Install with: pip install gradio") from e

try:
    from langchain_community.vectorstores import FAISS
    from langchain_community.embeddings import HuggingFaceEmbeddings
    from langchain_community.llms import HuggingFaceEndpoint
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    from langchain.memory import ConversationBufferMemory
except Exception as e:
    raise ImportError(
        "LangChain dependencies are required. Install with: pip install langchain langchain-community faiss-cpu sentence-transformers"
    ) from e

from langchain_core.documents import Document


In [ ]:
# Configure LLMs (Gemini primary, HF fallback)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

def build_gemini_model():
    if not genai:
        return None
    if not GOOGLE_API_KEY:
        return None
    try:
        genai.configure(api_key=GOOGLE_API_KEY)
        # Prefer the requested model name; fall back if the local library doesn't recognize it.
        for name in ["gemini-2.5-flash", "gemini-1.5-flash"]:
            try:
                return genai.GenerativeModel(name)
            except Exception:
                continue
        return None
    except Exception:
        return None

def build_hf_endpoint():
    if not HF_TOKEN:
        return None
    try:
        return HuggingFaceEndpoint(
            repo_id="mistralai/Mistral-7B-Instruct-v0.3",
            huggingfacehub_api_token=HF_TOKEN,
            temperature=0.2,
            max_new_tokens=512,
        )
    except Exception:
        return None

gemini_model = build_gemini_model()
hf_llm = build_hf_endpoint()

if gemini_model:
    print("Using Gemini.")
elif hf_llm:
    print("Using Hugging Face endpoint fallback.")
else:
    print(
        "No LLM configured. Set GOOGLE_API_KEY for Gemini, or HF_TOKEN for Hugging Face fallback, then re-run this cell."
    )


In [ ]:
# Build the knowledge base
CORPUS = [
    "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, and perception. Modern AI often relies on data-driven methods, particularly machine learning.",
    "Machine learning is a subfield of AI focused on algorithms that improve their performance on a task through experience. Supervised learning uses labeled examples, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning optimizes behavior through rewards.",
    "Large language models (LLMs) are machine learning models trained on massive text corpora to predict and generate language. They can be adapted for tasks like summarization, question answering, and coding assistance, but may hallucinate without grounding.",
    "Retrieval-augmented generation (RAG) combines information retrieval with generation. A system retrieves relevant passages from a knowledge base and conditions the model response on this context, improving factuality and making responses more traceable.",
]

docs = [Document(page_content=p, metadata={"source": f"doc{i}"}) for i, p in enumerate(CORPUS)]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

try:
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectordb = FAISS.from_documents(chunks, embeddings)
except Exception as e:
    raise RuntimeError(
        "Failed to build embeddings/vector store. This usually happens due to missing dependencies or a network issue when downloading the embedding model. "
        f"Original error: {e}"
    ) from e

retriever = vectordb.as_retriever(search_kwargs={"k": 3})
print("Vector store ready. Chunks:", len(chunks))


In [ ]:
# Conversation memory (trim to last 3 exchanges)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

SYSTEM_INSTRUCTIONS = (
    "You are a helpful assistant. Use the retrieved CONTEXT to answer the user's question. "
    "If the answer is not in the context, say you are not sure and answer conservatively. "
    "Keep answers concise and cite which doc sources you used (by source id)."
)

def trim_memory_last_3_exchanges():
    # Each exchange is typically (Human, AI) => 2 messages; keep last 6 messages.
    msgs = memory.chat_memory.messages
    if len(msgs) > 6:
        memory.chat_memory.messages = msgs[-6:]

def format_history() -> str:
    trim_memory_last_3_exchanges()
    parts = []
    for m in memory.chat_memory.messages:
        role = "User" if m.type == "human" else "Assistant"
        parts.append(f"{role}: {m.content}")
    return "\n".join(parts)

def retrieve_context(query: str) -> Tuple[str, List[str]]:
    docs = retriever.get_relevant_documents(query)
    sources = []
    ctx_parts = []
    for d in docs:
        sources.append(str(d.metadata.get("source", "unknown")))
        ctx_parts.append(f"[{sources[-1]}] {d.page_content}")
    return "\n\n".join(ctx_parts), sources

def call_llm(prompt: str) -> str:
    if gemini_model:
        try:
            resp = gemini_model.generate_content(prompt)
            return (resp.text or "").strip() or "(No response text)"
        except Exception as e:
            return f"Gemini call failed: {e}"
    if hf_llm:
        try:
            return str(hf_llm.invoke(prompt)).strip()
        except Exception as e:
            return f"Hugging Face endpoint call failed: {e}"
    return "No LLM configured. Set GOOGLE_API_KEY or HF_TOKEN and re-run configuration cells."


## Gradio Chat Demo
Example queries to try:
- "What is retrieval-augmented generation and why is it useful?"
- "What is the difference between supervised and reinforcement learning?"
- "Why can large language models hallucinate?"


In [ ]:
def rag_chat(message: str, history: List[List[str]]):
    if not message or not message.strip():
        return "Please enter a question."

    context, sources = retrieve_context(message)
    chat_history_text = format_history()

    prompt = textwrap.dedent(
        f"""
        {SYSTEM_INSTRUCTIONS}

        CONTEXT:
        {context}

        CHAT HISTORY (most recent last):
        {chat_history_text}

        USER QUESTION:
        {message}

        Answer:
        """
    ).strip()

    answer = call_llm(prompt)

    # Update LangChain memory
    memory.chat_memory.add_user_message(message)
    memory.chat_memory.add_ai_message(answer)
    trim_memory_last_3_exchanges()

    return answer

demo = gr.ChatInterface(
    fn=rag_chat,
    title="RAG Chatbot (LangChain + FAISS)",
    description=(
        "Grounds answers using a tiny custom corpus embedded with sentence-transformers and retrieved via FAISS. "
        "Uses Gemini if GOOGLE_API_KEY is set, otherwise falls back to a free Hugging Face endpoint if HF_TOKEN is set."
    ),
    examples=[
        "What is retrieval-augmented generation and why is it useful?",
        "What is the difference between supervised and reinforcement learning?",
        "Why can large language models hallucinate?",
    ],
)

demo.launch()
